# Oportunidades de inversión: activos artificialmente bajos y portafolios óptimos

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/santiagoriverti/oportunidades_inversion/blob/main/notebooks/01_oportunidades_y_portafolios.ipynb)

Sobre una lista de tickers que elegís vos, este notebook:

1. **Detecta qué activos están artificialmente bajos**: debajo de sus medias, de su tendencia y de su rango
   habitual, sobrevendidos, castigados contra el mercado y, para acciones, baratos según sus fundamentales
   (P/E contra su sector, PEG, flujo de caja, precio objetivo de los analistas).
2. **Arma 8 portafolios** que buscan el mejor retorno para cada nivel de riesgo (Markowitz, máximo Sharpe,
   mínima varianza, paridad de riesgo, HRP, Black-Litterman con las señales...) y te dice **qué comprar** con tu
   capital, en acciones/ETFs de EE.UU. o en **CEDEARs**.
3. **Prueba las estrategias en el pasado** (backtest) y **registra la corrida** en tu Google Drive para medir
   después cómo le fue de verdad (notebook 02).

**Cómo usarlo:** editá la celda de configuración (paso 1) y después *Entorno de ejecución → Ejecutar todas*.
Tarda 2 a 3 minutos con la lista por defecto (81 activos). Al final se descarga un ZIP con un Excel y un reporte
con todos los gráficos. Metodología completa: [docs/METODOLOGIA.md](https://github.com/santiagoriverti/oportunidades_inversion/blob/main/docs/METODOLOGIA.md).

> ⚠️ **Herramienta educativa y de análisis cuantitativo. No es asesoramiento financiero ni una recomendación de compra o venta.** Que un activo esté "barato" contra su historia no garantiza que vaya a subir, y los rendimientos pasados no garantizan rendimientos futuros.

In [ ]:
# Configuración: en Colab clona (o actualiza) el repo e instala la última yfinance; en la PC usa la carpeta del repo
import os, sys, subprocess, importlib
REPO = 'https://github.com/santiagoriverti/oportunidades_inversion.git'
if 'google.colab' in sys.modules:
    RAIZ = '/content/oportunidades_inversion'
    if os.path.exists(RAIZ):
        # Trae la última versión de GitHub y descarta cambios locales (la copia de Colab es descartable)
        subprocess.run(['git', '-C', RAIZ, 'fetch', '-q', '--depth', '1', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', RAIZ, 'reset', '-q', '--hard', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, RAIZ], check=True)
    # Yahoo cambia seguido: la última yfinance suele ser la que funciona
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'yfinance'], check=True)
else:  # subir desde la carpeta actual hasta encontrar el repo
    RAIZ = os.getcwd()
    while not os.path.exists(os.path.join(RAIZ, 'src', 'pipeline.py')) and os.path.dirname(RAIZ) != RAIZ:
        RAIZ = os.path.dirname(RAIZ)
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
# Si el notebook ya se corrió en esta sesión, olvidar los módulos de src cargados (pueden ser viejos)
for m in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[m]
importlib.invalidate_caches()
v = subprocess.run(['git', '-C', RAIZ, 'log', '-1', '--format=%h (%ad) %s', '--date=short'],
                   capture_output=True, text=True).stdout.strip()
print('Versión del repo:', v or 'sin git')

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import pandas as pd
from IPython.display import display, Markdown
from src.config import Config
from src.pipeline import Analisis
from src.portafolios import NOMBRES
from src import tablas, graficos, registro, expost, datos
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

## 1. Configuración

Lo único que hace falta tocar. Los tickers son los de **Yahoo Finance**: acciones y ETFs de EE.UU. tal cual
(`AAPL`, `SPY`), cripto con `-USD` (`BTC-USD`) y otras bolsas con su sufijo (`SAP.DE`, `IUIT.L`). Algunos
alias se corrigen solos (`BRKB` → `BRK-B`, `BTC` → `BTC-USD`). Todo se convierte a **dólares**.

Con `TICKERS = None` se usa la lista por defecto ([data/universo.csv](https://github.com/santiagoriverti/oportunidades_inversion/blob/main/data/universo.csv)),
que también dice qué tickers de la lista original se corrigieron o excluyeron y por qué.

In [ ]:
# ============================ 1. CONFIGURACIÓN (editá esta celda) ============================
# None = lista por defecto (81 activos). O tu propia lista, por ejemplo:
# TICKERS = ["AAPL", "MSFT", "NVDA", "GOOG", "GLD", "IEF", "BTC-USD", "SPY", "EWZ", "VALE"]
TICKERS = None

HORIZONTE = "mediano"         # "corto" (1-3 meses), "mediano" (6-12 meses) o "largo" (3+ años)
CAPITAL_USD = 10_000          # plata a invertir, para calcular cuánto comprar de cada activo
PESO_MAXIMO = 0.20            # tope por activo en cada portafolio (0.20 = 20%)
VOLATILIDAD_OBJETIVO = 0.10   # volatilidad anual buscada en el portafolio "Volatilidad objetivo"
INCLUIR_FUNDAMENTALES = True  # P/E, PEG, flujo de caja, precio objetivo de analistas (suma ~30 segundos)
CORRER_BACKTEST = True        # backtest walk-forward de todas las estrategias (~30 segundos)
GUARDAR_EN_DRIVE = True       # registra la corrida en Google Drive para el seguimiento (notebook 02)

cfg = Config.desde_horizonte(HORIZONTE, tickers=TICKERS, capital_usd=CAPITAL_USD, peso_maximo=PESO_MAXIMO,
                             volatilidad_objetivo=VOLATILIDAD_OBJETIVO, incluir_fundamentales=INCLUIR_FUNDAMENTALES,
                             correr_backtest=CORRER_BACKTEST, guardar_registro=GUARDAR_EN_DRIVE)
print(f"Horizonte {cfg.horizonte}: ventana de {cfg.ventana_anios:g} años, rebalanceo {cfg.rebalanceo}, "
      f"pesos del score {cfg.pesos_grupos}")

In [ ]:
A = Analisis(cfg).cargar_datos()
fuera = A.universo[A.universo["estado_analisis"] != "analizado"]
if len(fuera):
    display(Markdown("**Tickers que no se analizan** (y por qué):"))
    display(fuera[["ticker", "yahoo", "estado_analisis", "nota"]].set_index("ticker"))

## 2. ¿Qué está artificialmente bajo?

Para cada activo se calculan métricas en tres grupos y cada una se pasa a **percentil dentro de tu lista**
(100 = el más "barato"):

| Grupo | Métricas |
|---|---|
| **Técnico** | precio vs media de 50 y de 200 días, ubicación en el rango de 52 semanas, RSI de 14 días, %B de Bollinger |
| **Estadístico** | z-score del precio contra su año, desvío contra su tendencia de 3 años, qué tan inusual es la caída de 3 meses, z-score contra el benchmark (SPY) |
| **Fundamental** (solo acciones) | upside del precio objetivo de los analistas, P/E contra el ETF de su sector, PEG, rendimiento del flujo de caja libre |

El **score** (0-100) promedia los grupos con los pesos del horizonte. Como el score es *relativo* a la lista,
además se cuentan las **señales absolutas** (por ejemplo: precio debajo de su media de 200 días, RSI ≤ 30, caída
de 25% o más desde el máximo de 52 semanas, upside de analistas ≥ 20%).

- **Oportunidad**: score ≥ 70 y al menos 2 señales absolutas.
- **Oportunidad con alertas**: lo mismo, pero con una alerta: *cuchillo que cae* (−15% en un mes y tendencia
  bajista), *posible trampa de valor* (el mercado espera menores ganancias) o precio por encima del objetivo
  de los analistas. Mirarlas con más cuidado: lo barato puede seguir abaratándose.
- **Vigilar**, **Neutral**, **Sobreextendida** (score ≤ 30: está caro contra su propia historia).

In [ ]:
A.calcular_senales()
display(Markdown("### Oportunidades detectadas"))
display(tablas.estilo_senales(A.senales, n=None, solo=["Oportunidad", "Oportunidad con alertas"]))

In [ ]:
nombres = dict(zip(A.senales.index, A.senales["nombre"]))
graficos.fig_ranking(A.senales, 30, nombres).show()
graficos.fig_mapa_senales(A.senales, 25).show()

Precio de las oportunidades con sus medias de 50 y 200 días y las bandas de Bollinger (elegí el activo en el
menú de arriba a la derecha):

In [ ]:
from src.senales import top_oportunidades
graficos.fig_precios(A.precios, top_oportunidades(A.senales, 10)).show()

In [ ]:
display(Markdown("### Todos los activos (ordenados por score)"))
display(tablas.estilo_senales(A.senales, n=None))

In [ ]:
if A.fundamentales is not None:
    display(Markdown("### Fundamentales de las acciones (foto actual de Yahoo Finance)"))
    cols = ["sector", "pe_trailing", "pe_forward", "pe_relativo_sector", "peg", "fcf_yield", "rend_dividendo",
            "upside_analistas", "recomendacion", "n_analistas", "monedas_mezcladas"]
    f = A.fundamentales[A.fundamentales["tipo_yahoo"] == "EQUITY"][cols].sort_values("pe_relativo_sector")
    display(tablas.estilo_porcentajes(f, pct=["fcf_yield", "rend_dividendo", "upside_analistas"],
                                      dec={"pe_trailing": "{:.1f}", "pe_forward": "{:.1f}",
                                           "pe_relativo_sector": "{:.2f}", "peg": "{:.2f}",
                                           "recomendacion": "{:.1f}", "n_analistas": "{:.0f}"}))

## 3. Riesgo y retorno histórico

Métricas de cada activo en la ventana del horizonte (3 años en "mediano"): retorno anual compuesto,
volatilidad, **Sharpe** y **Sortino** (retorno extra sobre la tasa libre de riesgo por unidad de riesgo total o
de caída), **máxima caída**, **Calmar**, **VaR y CVaR** diarios al 95%, y **beta/alfa** contra el benchmark
(con retornos semanales, porque BTC y los ETFs europeos no cotizan en el mismo horario que EE.UU.).

In [ ]:
display(tablas.estilo_metricas(A.metricas.sort_values("sharpe", ascending=False)))

## 4. Portafolios sugeridos

Todos **sin ventas en corto**, con un **tope por activo** y con la covarianza estimada con retornos semanales y
contracción de **Ledoit-Wolf** (más estable). Los activos casi idénticos (*gemelos*: SPY/VOO/VTI, GLD/IAU,
QQQ/QQQM...) se optimizan como uno solo y se indican sus equivalentes.

| Portafolio | Idea |
|---|---|
| **Pesos iguales (1/N)** | referencia ingenua y difícil de vencer |
| **Mínima varianza** | el menor riesgo posible (no usa retornos esperados) |
| **Máximo Sharpe (Markowitz)** | el portafolio tangente: máximo retorno por unidad de riesgo |
| **Volatilidad objetivo** | máximo retorno con la volatilidad elegida en la configuración |
| **Paridad de riesgo** | cada activo aporta el mismo riesgo |
| **HRP (López de Prado)** | agrupa activos parecidos y reparte el riesgo por grupos; robusto a errores de estimación |
| **Black-Litterman con señales** | parte del equilibrio y lo inclina hacia los activos con mejor score de oportunidad |
| **Oportunidades** | las mejores oportunidades del paso 2, con paridad de riesgo |

> **Ojo con los retornos esperados:** Markowitz, máximo Sharpe y volatilidad objetivo usan el retorno
> histórico de la ventana, que premia a los que *ya* subieron mucho. Por eso el resumen muestra también la
> versión histórica de cada portafolio, y conviene mirar los robustos (mínima varianza, paridad de riesgo, HRP) y
> el backtest del paso 6 antes de decidir.

In [ ]:
A.optimizar()
display(Markdown("### Resumen (ex-ante con el modelo; *_hist* = cómo le fue en la ventana con esos pesos)"))
display(tablas.estilo_resumen_portafolios(A.resumen.drop(columns=["clave"])))
if len(A.tabla_gemelos):
    display(Markdown("### Gemelos: se optimiza con el primero; los otros son equivalentes"))
    display(A.tabla_gemelos)
display(Markdown("### Activos que quedaron fuera de la optimización"))
display(A.excluidos_opt)

In [ ]:
res = A.portafolios
graficos.fig_frontera(res["frontera"], res["mu"], res["cov"], A.pesos, A.rf).show()
graficos.fig_pesos(A.pesos).show()
graficos.fig_contribucion_riesgo(A.pesos, res["cov"]).show()

In [ ]:
from src.portafolios import orden_hrp
graficos.fig_correlaciones(res["cov"], orden_hrp(res["cov"])).show()

## 5. Qué comprar

Cantidades para el `CAPITAL_USD` de la configuración, en dos versiones:

- **EE.UU.**: acciones y ETFs originales al último precio (las cripto se compran fraccionadas).
- **CEDEARs**: especie D (dólar MEP) de BYMA con el precio en vivo de [data912](https://data912.com); si un CEDEAR
  solo cotiza en pesos, se pasa a dólares con el **MEP implícito**. Lo que no tiene CEDEAR se marca para comprar
  afuera.

Se compran unidades enteras: primero la parte entera de cada posición y después, con lo que sobra, la posición
más lejos de su objetivo. Elegí el portafolio en la celda de abajo.

In [ ]:
A.asignar()
# Elegí uno: pesos_iguales, minima_varianza, maximo_sharpe, volatilidad_objetivo, paridad_riesgo, hrp,
#            black_litterman, oportunidades
PORTAFOLIO = "maximo_sharpe"

t, sobrante = A.tabla_compra(PORTAFOLIO, "usd")
display(Markdown(f"### {NOMBRES[PORTAFOLIO]} — en EE.UU. (sobran USD {sobrante:,.2f})"))
display(tablas.estilo_asignacion(t[t["peso_objetivo"] > 0]))
if "cedear" in A.asignaciones[PORTAFOLIO]:
    t2, sobrante2 = A.tabla_compra(PORTAFOLIO, "cedear")
    display(Markdown(f"### {NOMBRES[PORTAFOLIO]} — con CEDEARs (sobran USD {sobrante2:,.2f}; "
                     f"MEP implícito {A.mep:,.0f})" if A.mep else f"### {NOMBRES[PORTAFOLIO]} — con CEDEARs"))
    display(tablas.estilo_asignacion(t2))

## 6. Backtest: ¿cómo les hubiera ido a las estrategias?

**Walk-forward**: en cada fin de trimestre (en el horizonte "mediano") se recalculan las señales y los
portafolios **solo con los datos disponibles hasta ese día**, se mantienen hasta el próximo rebalanceo y se
descuenta un costo por rotación. Además se mide si el **score anticipó** los retornos del trimestre siguiente
(IC de Spearman: 0 = nada, positivo = los de score alto rindieron más).

> **Sesgos a tener en cuenta:** la lista de tickers se eligió hoy (no están los que quebraron o se dejaron de
> mirar: *sesgo de supervivencia*), y el score del backtest no usa fundamentales (Yahoo no da su historia).
> Tomalo como una referencia de comportamiento, no como una promesa.

In [ ]:
if cfg.correr_backtest:
    A.correr_backtest()
    display(tablas.estilo_backtest(A.backtest["metricas"]))
    graficos.fig_backtest(A.backtest["retornos"], A.benchmark).show()
    ic = A.resumen_ic
    if ic:
        display(Markdown(
            f"**Capacidad predictiva del score** en {ic['periodos']} rebalanceos: IC medio **{ic['ic_medio']:+.3f}** "
            f"(t = {ic['ic_t_stat']:.2f}; positivo en {ic['pct_ic_positivo']:.0%} de los períodos). Las oportunidades "
            f"rindieron en promedio **{ic['exceso_medio_oportunidades']:+.1%}** por período contra el promedio de la "
            f"lista y le ganaron en {ic['pct_periodos_oportunidades_ganan']:.0%} de los períodos. "
            "Un |t| menor a 2 quiere decir que no se distingue del azar."))
        graficos.fig_ic(A.backtest["ic"]).show()
else:
    print("Backtest desactivado en la configuración (CORRER_BACKTEST = False).")

## 7. Guardar la corrida (para el seguimiento ex-post)

Se guarda en **Google Drive**, en `Mi unidad/oportunidades_inversion/registros/AAAAMMDD_HHMMSS/` (la primera vez
Colab pide permiso para acceder al Drive). Ahí quedan las señales, los pesos de los 8 portafolios, las cantidades a
comprar y los precios del día. El **notebook 02** lee todas las corridas y mide cuánto rindió cada estrategia.

In [ ]:
if cfg.guardar_registro:
    A.guardar()
else:
    print("No se guarda la corrida (GUARDAR_EN_DRIVE = False).")

## 8. Descargar todo

ZIP con el **Excel** (una hoja por tabla), el **reporte HTML** con todos los gráficos interactivos (se abre sin
internet) y la carpeta de la corrida. En Colab se descarga solo.

Para ver cómo les fue a las corridas guardadas: [![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/santiagoriverti/oportunidades_inversion/blob/main/notebooks/02_seguimiento_expost.ipynb)

In [ ]:
zip_path = A.exportar()
if A.avisos:
    display(Markdown("**Avisos de esta corrida:**\n" + "\n".join(f"- {a}" for a in A.avisos)))